In [1]:
# Supervised + Unsupervised ML

In [2]:
# Dataset - Customer Churn Prediction

Business Problem : -
A telecom company wants to predict which customers will churn (cancel their subscription). They also want to segment customers into groups for targeted marketing.



Technique -

In [ ]:
# Technique	                                           
# Data Preprocessing & Feature Engineering	            
# Classification (Supervised Learning)	                
# Clustering (Unsupervised Learning)	                  
# Dimensionality Reduction (PCA)	                      
# Ensemble/Boosting (XGBoost)	                          
# Hyperparameter Tuning & Pipelines	                    


In [17]:
# Download from Kaggle or use this synthetic version
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
import kagglehub
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

In [ ]:
path

In [ ]:
import os

# Construct the full path to the CSV file
file_path = os.path.join(path, 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
display(df.head())

In [ ]:
print(f"Dataset Shape: {df.shape}")
print(f"\nTarget Distribution:\n{df['Churn'].value_counts(normalize=True)}")
df.head()

In [ ]:
#Phase 1: Exploratory Data Analysis (EDA)

In [ ]:
# ============================
# 1.1 Basic Info
# ============================
print(f"Shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nDescribe:\n{df.describe()}")

# ============================
# 1.2 Target Variable Analysis
# ============================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Churn Distribution (Count)')
df['Churn'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Churn Distribution (%)')
plt.tight_layout()
plt.show()

# ============================
# 1.3 Feature Analysis
# ============================
# Numeric features
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5*len(numeric_cols), 4))
for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='Churn', ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

# Categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.drop('Churn')
for col in categorical_cols[:6]:  # first 6
    pd.crosstab(df[col], df['Churn'], normalize='index').plot(kind='bar', stacked=True)
    plt.title(f'{col} vs Churn')
    plt.show()

In [ ]:
# Phase 2: Data Preprocessing

In [ ]:
# ============================
# 2.1 Clean Data
# ============================

# Ensure df is loaded from the start of this processing if it's currently empty
# This handles cases where previous steps might have cleared df or it was not run
import pandas as pd
import numpy as np
import os

if 'df' not in locals() or df.empty:
    print("DataFrame 'df' is empty or not found in current scope. Attempting to reload data from source.")
    try:
        # Assuming 'path' variable is correctly set by a previous cell
        # (e.g., F6yZcoefEQC1 and 7rAHXysnF-Hr provide `path = '/kaggle/input/telco-customer-churn'`)
        file_path = os.path.join(path, 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
        df = pd.read_csv(file_path)
        print(f"Data reloaded. Shape: {df.shape}")
    except NameError:
        raise NameError("The 'path' variable is not defined. Please run previous cells (e.g., F6yZcoefEQC1) to download the dataset.")
    except Exception as e:
        raise RuntimeError(f"Error reloading data: {e}. Please ensure data loading cells are correctly executed.")


# Convert TotalCharges to numeric (may have spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop customerID (not useful for modeling)
# Check if 'customerID' column exists before dropping it
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

# Encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop rows with NaN values from critical columns:
# 'TotalCharges' (introduced by coerce) and 'Churn' (if the mapping somehow introduced them).
# This ensures both features and target are clean before splitting.
df.dropna(subset=['TotalCharges', 'Churn'], inplace=True)

# Check if DataFrame is empty after preprocessing
if df.empty:
    raise ValueError("DataFrame became empty after cleaning 'TotalCharges' and 'Churn' NaNs. This indicates an issue with the dataset or cleaning logic that removed all rows.")

# ============================
# 2.2 Define Features and Target
# ============================
X = df.drop('Churn', axis=1)
y = df['Churn']

# Identify column types
numeric_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric Features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical Features ({len(categorical_features)}): {categorical_features}")

# ============================
# 2.3 Train-Test Split
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print(f"Train Churn Rate: {y_train.mean():.2%}")
print(f"Test Churn Rate: {y_train.mean():.2%}")

In [ ]:
# Phase 3: Build ML Pipeline

In [ ]:
# ============================
# 3.1 Preprocessing Pipeline
# ============================
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), #mode
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# ============================
# 3.2 Compare Multiple Models
# ============================
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42,
                                  use_label_encoder=False, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = {
        'mean': scores.mean(),
        'std': scores.std()
    }
    print(f"{name:25s} → Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

# Visualize comparison
plt.figure(figsize=(10, 5))
plt.bar(results.keys(), [r['mean'] for r in results.values()],
        yerr=[r['std'] for r in results.values()], capsize=5)
plt.ylabel('Accuracy')
plt.title('Model Comparison')
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Phase 4: Hyperparameter Tuning (Best Model)

In [ ]:
# ============================
# 4.1 Tune XGBoost
# ============================
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    xgb_pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best ROC-AUC: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_


In [ ]:
# Phase 5: Model Evaluation

In [ ]:
# ============================
# 5.1 Predictions
# ============================
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# ============================
# 5.2 Metrics
# ============================
print("=" * 50)
print("FINAL MODEL EVALUATION")
print("=" * 50)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# ============================
# 5.3 Confusion Matrix
# ============================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# ============================
# 5.4 ROC Curve
# ============================
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', label=f'XGBoost (AUC = {roc_auc_score(y_test, y_pred_proba):.4f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

# ============================
# 5.5 Feature Importance
# ============================
# Get feature names after preprocessing
feature_names = (numeric_features +
                 list(best_model.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .named_steps['onehot']
                      .get_feature_names_out(categorical_features)))

importances = best_model.named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
feat_imp[:15].plot(kind='barh')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.show()


In [ ]:
# Phase 6: Customer Segmentation (Unsupervised)

In [ ]:
# sinua

In [ ]:
# ============================
# 6.1 Preprocess for Clustering
# ============================
X_processed = preprocessor.fit_transform(X)

# ============================
# 6.2 PCA for Visualization
# ============================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)
print(f"Variance captured by 2 PCs: {sum(pca.explained_variance_ratio_):.2%}")

# ============================
# 6.3 Find Optimal K (Elbow Method)
# ============================
inertias = []
K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_processed)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

# ============================
# 6.4 K-Means Clustering
# ============================
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_processed)

# Visualize clusters in PCA space
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', alpha=0.5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Customer Segments (K-Means, K=4)')
plt.colorbar(scatter, label='Cluster')
plt.show()

# ============================
# 6.5 Analyze Segments
# ============================
df['Cluster'] = clusters
for i in range(4):
    print(f"\n{'='*40}")
    print(f"CLUSTER {i} (n={sum(clusters==i)})")
    print(f"{'='*40}")
    cluster_data = df[df['Cluster'] == i]
    print(f"Churn Rate: {cluster_data['Churn'].mean():.2%}")
    for col in numeric_features:
        print(f"{col}: {cluster_data[col].mean():.2f}")


In [ ]:
import joblib

# Save the complete pipeline
joblib.dump(best_model, 'churn_prediction_pipeline.pkl')
print("✅ Model saved successfully!")

# To load and use later:
# loaded_model = joblib.load('churn_prediction_pipeline.pkl')
# new_predictions = loaded_model.predict(new_data)

📊 Project Summary & Presentation Points

In [ ]:
# Key Findings to Discuss
# Which model performed best and why?
# Top 5 features driving churn
# Customer segments discovered — which segment has highest churn?
# Business recommendations based on the analysis
# Model accuracy and reliability (cross-validation scores)


# What This Project Demonstrates


# ✅ End-to-end ML workflow
# ✅ EDA and data preprocessing
# ✅ Multiple model comparison
# ✅ Hyperparameter tuning with cross-validation
# ✅ Proper pipeline usage (no data leakage)
# ✅ Both supervised (classification) and unsupervised (clustering) techniques
# ✅ Dimensionality reduction (PCA)
# ✅ Model saving for deployment